# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [1]:
from asyncio import timeout

import requests
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

def fetch_pokemon(poke_id, session):
    response = session.get(f"https://pokeapi.co/api/v2/pokemon/{poke_id}",
    timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    return {
        "name": data["name"],
        "height": data["height"],
        "weight": data["weight"],
        "base_experience": data["base_experience"],
        "first_type": data["types"][0]["type"]["name"],
    }

with requests.Session() as session:
    pokemon = [fetch_pokemon(i, session) for i in range(1, 152)]

print(f"Fetched {len(pokemon)} Pokémon.")

Fetched 151 Pokémon.


In [2]:
df = pd.DataFrame(pokemon)

df["weight_kg"] = df["weight"] / 10

df.head()

,name,height,weight,base_experience,first_type,weight_kg
0,bulbasaur,7,69,64,grass,6.9
1,ivysaur,10,130,142,grass,13.0
2,venusaur,20,1000,236,grass,100.0
3,charmander,6,85,62,fire,8.5
4,charmeleon,11,190,142,fire,19.0


In [3]:
#Q1. Heaviest Pokémon: Snorlax - 460kg
df.nlargest(5, "weight_kg")[["name", "weight_kg"]]

#Q2. Average Weight - 46kg
df["weight_kg"].mean().round(1)

#Q3. Count per type - Water, Normal and Poison
df["first_type"].value_counts()

#Q4. Average weight per type
(df.groupby("first_type")["weight_kg"].agg(["mean","count"]).round(2)).sort_values("mean", ascending=False)


,mean,count
first_type,,
rock,87.61,9
dragon,76.60,3
water,57.97,28
fighting,54.29,7
psychic,51.56,8
normal,50.09,22
fire,48.02,12
ice,48.00,2
ground,45.26,8


## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

In [4]:
# Your code goes here

# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [5]:
url = "https://api.coingecko.com/api/v3/simple/price"
headers = {"Authorization":f"Bearer {os.environ['CG-JYaHKNH3Zg36qbBEDvar7Hak']}"}

params = {
    "vs_currencies": "eur",
    "ids": "bitcoin,ethereum,solana,dogecoin"
}


response = requests.get(url, params=params, headers=headers, timeout=10)
response.raise_for_status()
data_gecko = response.json()

data_gecko

KeyError: 'CG-JYaHKNH3Zg36qbBEDvar7Hak'

In [ ]:
crypto_df = pd.DataFrame(data_gecko).T.reset_index().rename(columns={"index":"coins", "eur":"price"})

cond_price = [
    crypto_df["price"] > 1000,
    crypto_df["price"].between(100, 1000),
    crypto_df["price"] < 100
]

categories = ["high", "medium", "low"]

crypto_df["price_category"] = np.select(cond_price,categories, default="unknown")

crypto_df["price_rank"] = crypto_df["price"].rank(ascending=False).astype(int)


In [ ]:
#Q1. Most expensive coin
crypto_df.nlargest(1, "price")[["coins", "price"]]

In [ ]:
#Q2. Category with most coins
crypto_df["price_category"].value_counts()

In [ ]:
#Bonus - price vs bitcoin
btc_price = crypto_df.loc[crypto_df["coins"] == "bitcoin", "price"].iat[0]
crypto_df["btc_category"] = (crypto_df["price"] / btc_price * 100).round(2)
crypto_df[["coins", "price", "btc_category"]]